# REPSOL Model Training (70/15/15)

This notebook runs the full pipeline: 
1. Configure training hyperparameters.
2. Verify spectrogram tensor files in train/val/test.
3. Train EfficientNet with live progress bars.
4. Evaluate on validation and test sets.
5. Print detailed classification report and confusion matrix.

In [ ]:
from pathlib import Path
import torch

# ===== Hyperparameters (edit these) =====
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-3
PATIENCE = 4
MODEL_NAME = "efficientnet"

# ===== Paths =====
PROJECT_ROOT = Path(r"D:\Internships\INMAR\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms"
MODELS_DIR = PROJECT_ROOT / "Models_files"
CHECKPOINT_PATH = MODELS_DIR / f"{MODEL_NAME}_best_01.pth"
HISTORY_PATH = MODELS_DIR / f"{MODEL_NAME}_best_training_history_01.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPECTROGRAM_DIR:", SPECTROGRAM_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("HISTORY_PATH:", HISTORY_PATH)
print("DEVICE:", DEVICE)
print("BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS, "| LR:", LEARNING_RATE)

PROJECT_ROOT: C:\home\ben\REPSOL
SPECTROGRAM_DIR: C:\home\ben\REPSOL\Data\Spectrograms
CHECKPOINT_PATH: C:\home\ben\REPSOL\efficientnet_best.pth
DEVICE: cpu
BATCH_SIZE: 16 | EPOCHS: 15 | LR: 0.001


In [10]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("PT files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .pt files found."
assert counts["val"] > 0, "No val .pt files found."
assert counts["test"] > 0, "No test .pt files found."

PT files by split: {'train': 1383, 'val': 296, 'test': 297}
Total: 1976


In [11]:
# Install/verify training dependencies in the active notebook kernel
import importlib
import subprocess
import sys

required = ["torch", "torchvision", "torchaudio", "scikit-learn", "pandas", "tqdm", "numpy"]
name_map = {
    "scikit-learn": "sklearn",
}

for pkg in required:
    import_name = name_map.get(pkg, pkg.replace("-", "_"))
    try:
        importlib.import_module(import_name)
        print(f"OK: {pkg}")
    except Exception:
        print(f"Installing: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed: {pkg}")

OK: torch
Installing: torchvision
Installed: torchvision
OK: torchaudio
OK: scikit-learn
OK: pandas
OK: tqdm
OK: numpy


In [ ]:
import importlib
import torch

try:
    import torchvision  # noqa: F401
except Exception as e:
    raise RuntimeError(
        "torchvision is not available in this notebook kernel. "
        "Run the dependency cell right above this one, then retry."
    ) from e

import EfficientNet.train as train_module

train_module = importlib.reload(train_module)
Trainer = train_module.Trainer

trainer = Trainer(
    spectrogram_dir=SPECTROGRAM_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    lr=LEARNING_RATE,
    device=DEVICE,
)

trainer.fit()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:01<00:00, 17.0MB/s]


Train samples: 1383
Train batches: 87


Epoch 1/15 Validation: 100%|████████████████████████| 19/19 [00:02<00:00,  6.42batch/s, loss=6.1628]

Epoch 1/15 | Train Loss: 1.4865 | Val Loss: 1.6850 | Train Acc: 44.32 | Val Acc: 49.32


...

Epoch 8/15 | Train Loss: 0.4313 | Val Loss: 1.2318 | Train Acc: 79.25 | Val Acc: 60.47
No improvement for 2/4 epochs



Epoch 9/15 Validation: 100%|████████████████████████| 19/19 [00:02<00:00,  6.87batch/s, loss=1.1414]

Epoch 9/15 | Train Loss: 0.3272 | Val Loss: 1.3456 | Train Acc: 83.73 | Val Acc: 63.51
No improvement for 3/4 epochs



Epoch 10/15 Validation: 100%|███████████████████████| 19/19 [00:02<00:00,  6.75batch/s, loss=1.5983]

Epoch 10/15 | Train Loss: 0.1858 | Val Loss: 1.0221 | Train Acc: 89.88 | Val Acc: 69.59
No improvement for 4/4 epochs
Early stopping: no improvement for 4 epochs.
Training finished.


In [13]:
import importlib
import torch
import src.evaluate as eval_module

eval_module = importlib.reload(eval_module)
evaluate_model = eval_module.evaluate_model

state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
trainer.model.load_state_dict(state)

val_metrics = evaluate_model(trainer.model, trainer.val_loader, DEVICE)
test_metrics = evaluate_model(trainer.model, trainer.test_loader, DEVICE)

print("Validation Metrics")
print({
    "accuracy": round(val_metrics["accuracy"], 4),
    "precision": round(val_metrics["precision"], 4),
    "recall": round(val_metrics["recall"], 4),
    "f1": round(val_metrics["f1"], 4),
})

print("\nTest Metrics")
print({
    "accuracy": round(test_metrics["accuracy"], 4),
    "precision": round(test_metrics["precision"], 4),
    "recall": round(test_metrics["recall"], 4),
    "f1": round(test_metrics["f1"], 4),
})

Validation Metrics
{'accuracy': 0.6993, 'precision': 0.7217, 'recall': 0.6993, 'f1': 0.6811}

Test Metrics
{'accuracy': 0.6768, 'precision': 0.69, 'recall': 0.6768, 'f1': 0.6548}


In [14]:
print("Test Classification Report:\n")
print(test_metrics["report"])

print("Test Confusion Matrix:")
print(test_metrics["confusion_matrix"])

Test Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.73      0.81        33
           1       0.43      0.50      0.46         6
           2       0.47      0.80      0.59        20
           3       0.71      0.63      0.67       106
           4       0.60      0.15      0.24        39
           5       1.00      1.00      1.00        11
           6       0.66      0.93      0.77        76
           7       0.43      0.50      0.46         6

    accuracy                           0.68       297
   macro avg       0.65      0.66      0.63       297
weighted avg       0.69      0.68      0.65       297

Test Confusion Matrix:
[[24  0  0  0  2  0  7  0]
 [ 0  3  0  1  0  0  2  0]
 [ 0  0 16  4  0  0  0  0]
 [ 2  2  7 67  2  0 23  3]
 [ 0  0 11 18  6  0  4  0]
 [ 0  0  0  0  0 11  0  0]
 [ 0  1  0  3  0  0 71  1]
 [ 0  1  0  1  0  0  1  3]]
